# Demo: Doctor-Hospital Rank Assignment

Pipeline: `input_generation.py` (created by Ansh) -> `interfernce_function_june.py` (created by June) -> `Optimizor.py` (created by James)

This script first creates some mock rankings and hospital capacities, converts rankings to costs via
the interference function `g()`, then finds assignment that minimizes total cost subject to capacity 
constraints, using min-cost flow.

In [1]:
import numpy as np
from input_generation import generate_mock_dataset
from interference_function import g
from Optimizer import minimum_cost

In [2]:
N_DOCTORS = 20
N_HOSPITALS = 5
SIGMA = 1.0     # popularity skew -- higher = more lopsided hospital demand
SEED = 42       # for reproducibility

preferences, capacities = generate_mock_dataset(
    n_doctors=N_DOCTORS, n_hospitals=N_HOSPITALS, sigma=SIGMA, seed=SEED
)

print("preferences (d x h):", preferences.shape)
print(preferences)
print("\ncapacities:", capacities)
print("total capacity:", capacities.sum(), " | doctors:", N_DOCTORS)

preferences (d x h): (20, 5)
[[4 3 2 1 5]
 [3 5 1 2 4]
 [3 1 2 4 5]
 [5 2 4 1 3]
 [1 3 2 4 5]
 [3 5 4 1 2]
 [1 4 2 3 5]
 [3 4 1 2 5]
 [3 2 4 1 5]
 [1 2 4 3 5]
 [3 4 2 1 5]
 [3 5 2 1 4]
 [3 4 1 2 5]
 [4 1 2 3 5]
 [2 4 1 3 5]
 [4 2 3 1 5]
 [1 4 3 2 5]
 [1 4 3 2 5]
 [3 2 4 1 5]
 [3 5 2 1 4]]

capacities: [5 3 3 3 6]
total capacity: 20  | doctors: 20


In [3]:
avg_rank = preferences.mean(axis=0)
for j in range(N_HOSPITALS):
    print(f"Hospital {j+1}: avg rank {avg_rank[j]:.2f}, capacity {capacities[j]}")

Hospital 1: avg rank 2.70, capacity 5
Hospital 2: avg rank 3.30, capacity 3
Hospital 3: avg rank 2.45, capacity 3
Hospital 4: avg rank 1.95, capacity 3
Hospital 5: avg rank 4.60, capacity 6


In [4]:
costs = g(preferences)
print("costs == preferences:", np.array_equal(costs, preferences))

costs == preferences: True


In [5]:
assignments, total_cost = minimum_cost(costs, capacities)

print(f"Total assignment cost was: {total_cost}")
print(f"Average rank achieved was: {total_cost / N_DOCTORS:.2f}")
print()
print("Matchings are below (doctors who got their favorite ranks at the top, \ndoctors who got least favorite ranks at the bottom):")
print()
for doctor in sorted(assignments, key=lambda d: (preferences[d, assignments[d]], d)):
    hospital = assignments[doctor]
    rank_received = preferences[doctor, hospital]
    print(f"Doctor {doctor+1} matched to Hospital {hospital+1}  (doctor got their #{rank_received} choice)")

Total assignment cost was: 37
Average rank achieved was: 1.85

Matchings are below (doctors who got their favorite ranks at the top, 
doctors who got least favorite ranks at the bottom):

Doctor 1 matched to Hospital 4  (doctor got their #1 choice)
Doctor 3 matched to Hospital 2  (doctor got their #1 choice)
Doctor 5 matched to Hospital 1  (doctor got their #1 choice)
Doctor 7 matched to Hospital 1  (doctor got their #1 choice)
Doctor 8 matched to Hospital 3  (doctor got their #1 choice)
Doctor 10 matched to Hospital 1  (doctor got their #1 choice)
Doctor 11 matched to Hospital 4  (doctor got their #1 choice)
Doctor 13 matched to Hospital 3  (doctor got their #1 choice)
Doctor 14 matched to Hospital 2  (doctor got their #1 choice)
Doctor 15 matched to Hospital 3  (doctor got their #1 choice)
Doctor 17 matched to Hospital 1  (doctor got their #1 choice)
Doctor 18 matched to Hospital 1  (doctor got their #1 choice)
Doctor 19 matched to Hospital 4  (doctor got their #1 choice)
Doctor 6 ma

Confirm every doctor is assigned exactly once and no hospital exceeds capacity.

In [6]:
assert len(assignments) == N_DOCTORS, "some doctor was not assigned"

hospital_load = np.zeros(N_HOSPITALS, dtype=int)
for hospital in assignments.values():
    hospital_load[hospital] += 1

assert (hospital_load <= capacities).all(), "some hospital exceeded the capacity"

print("All doctors assigned once exactly.")
print("Hospital load vs capacity:")
for j in range(N_HOSPITALS):
    print(f"  Hospital {j+1}: {hospital_load[j]} / {capacities[j]}")

All doctors assigned once exactly.
Hospital load vs capacity:
  Hospital 1: 5 / 5
  Hospital 2: 3 / 3
  Hospital 3: 3 / 3
  Hospital 4: 3 / 3
  Hospital 5: 6 / 6


You can test with any other mock input here if you like. Use the below code snippet

In [7]:
# preferences = np.array([...])  # (d doctors x h hospitals), entry = rank (1 = best)
# capacities = np.array([...])    # (h,) hospital capacities, sum >= d doctors

# costs = g(preferences)
# assignments, total_cost = minimum_cost(costs, capacities)